# Week 5: CUDA preprocessing benchmark on Colab

In Colab, set **Runtime → Change runtime type → T4 GPU** before running. Put this repository in Google Drive at `MyDrive/vision-engine`, or edit `REPO_DIR` in the setup cell. This notebook builds the optional CUDA path, runs CTest including CUDA parity/fallback checks, and saves raw timings in `bench/preprocess/colab_t4.json` on Drive.

CUDA timing includes host-to-device and device-to-host copies using preallocated device buffers. It measures the full preprocessing call, not only kernel time. Do not compare GPU timings from a CPU runtime or claim a CUDA result unless `cuda_active` is true in the saved JSON.

In [3]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess

drive.mount("/content/drive")

REPO_URL = "https://github.com/elgunismayil0v2001/vision-engine.git"
REPO_DIR = Path("/content/vision-engine")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if not (REPO_DIR / "cpp/CMakeLists.txt").is_file():
    raise FileNotFoundError(f"cpp/CMakeLists.txt not found in {REPO_DIR}. Is it pushed to GitHub?")

BUILD_DIR = Path("/content/vision-engine-build/cpp-cuda")
LOG_PATH = REPO_DIR / "bench/preprocess/colab_t4.json"
DRIVE_OUT = Path("/content/drive/MyDrive/vision-engine-results")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)
print("Build:", BUILD_DIR)
print("Raw log:", LOG_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/vision-engine
Build: /content/vision-engine-build/cpp-cuda
Raw log: /content/vision-engine/bench/preprocess/colab_t4.json


In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select a Colab T4 runtime and reconnect.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

RuntimeError: No CUDA GPU is attached. Select a Colab T4 runtime and reconnect.

## Install the pinned parity runtime and ONNX Runtime C++ SDK

Colab provides the CUDA toolkit and PyTorch. This installs only the existing pinned ONNX Python packages used by the CTest parity target, then downloads the project-pinned Linux CPU C++ SDK required by the engine target.

In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

required = {"onnx": "1.20.1", "onnxruntime": "1.31.1"}
needs_install = False
for package, version in required.items():
    try:
        needs_install |= importlib.metadata.version(package) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *[f"{name}=={version}" for name, version in required.items()]], check=True)

ORT_VERSION = "1.31.1"
ORT_ROOT = Path("/content/onnxruntime-sdk")
ORT_LIBRARY = ORT_ROOT / "lib/libonnxruntime.so"
if not ORT_LIBRARY.is_file():
    archive = Path("/content/onnxruntime-linux-x64.tgz")
    url = f"https://github.com/microsoft/onnxruntime/releases/download/v{ORT_VERSION}/onnxruntime-linux-x64-{ORT_VERSION}.tgz"
    subprocess.run(["curl", "-fL", url, "-o", str(archive)], check=True)
    ORT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(archive), "--strip-components=1", "-C", str(ORT_ROOT)], check=True)
print("ONNX Runtime SDK:", ORT_ROOT)

## Configure, build, and test

The free Colab T4 uses compute capability 7.5. The CUDA option is enabled only in this build; normal Mac and CI builds keep it off.

In [ ]:
configure = [
    "cmake", "-S", str(REPO_DIR / "cpp"), "-B", str(BUILD_DIR),
    f"-DONNXRUNTIME_ROOT={ORT_ROOT}",
    f"-DPython3_EXECUTABLE={sys.executable}",
    "-DENABLE_CUDA=ON", "-DENABLE_OPENMP=ON",
    "-DCMAKE_CUDA_ARCHITECTURES=75", "-DBUILD_TESTING=ON",
]
subprocess.run(configure, check=True)
subprocess.run(["cmake", "--build", str(BUILD_DIR), "-j2"], check=True)
subprocess.run(["ctest", "--test-dir", str(BUILD_DIR), "--output-on-failure"], check=True)

## Measure and persist raw results

The benchmark uses a deterministic 640×480 RGB input, 50 warmups, 500 timed iterations, and four OpenMP threads. Change the values only if you record them with the run. It verifies each result against the scalar reference before writing the JSON.

In [ ]:
import json
benchmark = [
    sys.executable, str(REPO_DIR / "bench/preprocess/run_benchmark.py"),
    "--binary", str(BUILD_DIR / "preprocess_benchmark"),
    "--output", str(LOG_PATH),
    "--width", "640", "--height", "480",
    "--warmup", "50", "--iterations", "500", "--threads", "4",
]
subprocess.run(benchmark, cwd=REPO_DIR, check=True)
result = json.loads(LOG_PATH.read_text())
if not result["metadata"]["cuda_active"]:
    raise RuntimeError("The saved run used the CPU fallback, not CUDA.")
for row in result["results"]:
    print(f"{row['kernel']:24s} p50={row['p50_ms']:.4f} ms "
          f"p99={row['p99_ms']:.4f} ms "
          f"speedup={row['speedup_vs_scalar']:.3f}x "
          f"max_abs_error={row['max_abs_error']:.2g}")
print("Saved:", LOG_PATH)

In [ ]:
shutil.copy(LOG_PATH, DRIVE_OUT / "colab_t4.json")
print("Saved to", DRIVE_OUT)